In [0]:
%run ../init_framework_silver

In [0]:
# Execute the read logic
df_lkp_raw = read_reference_source(
    spark=spark, 
    source_path=LOAN_PURPOSE_PATH, 
    schema=LOAN_PURPOSES_SCHEMA
)

# 2. Enrich with Silver Metadata
df_lkp_enriched = add_ref_metadata(df_lkp_raw)

# Create Temp View
df_lkp_enriched.createOrReplaceTempView("tmp_loan_purpose_seed")

# Execute MERGE
spark.sql(
    f"""
    MERGE INTO {REF_LOAN_PURPOSES} AS target
    USING tmp_loan_purpose_seed AS source
    ON target.loan_purpose = source.loan_purpose
    
    WHEN MATCHED AND (target.is_active <> source.is_active) THEN
      UPDATE SET 
        target.is_active = source.is_active,
        target._updated_at = source._updated_at,
        target._source_file = source._source_file,
        target._processed_by = source._processed_by
        
    WHEN NOT MATCHED THEN
      INSERT (
        loan_purpose, 
        is_active,         
        _updated_at,
        _source_file, 
        _processed_by
      ) 
      VALUES (
        source.loan_purpose, 
        source.is_active, 
        source._updated_at,       
        source._source_file, 
        source._processed_by
      )
"""
)

In [0]:
%sql
select * from   lending_risk.silver.ref_loan_purposes